# Evaluate itn sle control


In [ ]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from joblib import dump, load
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score, confusion_matrix, f1_score, balanced_accuracy_score

MODEL_DIR = 'results/classification_sle_external/results_elasticnet_C1_l1_0.5'

In [ ]:
adata_itn = ad.read_h5ad('data/adata_cohort_2.h5ad')
adata_itn.obs = adata_itn.obs.rename(columns={'sample_ID': 'patient_ID'})

print("ITN validation set:", adata_itn.shape)
print(adata_itn.obs['group'].value_counts(dropna=False))
n_sle = int((adata_itn.obs['group'] == 'lupus').sum())
n_hc = int((adata_itn.obs['group'] == 'healthy_control').sum())
print(f"\n{n_sle} SLE (lupus_ITN) samples, {n_hc} HC samples (expected 297 / 91)")
print("Unique donors:", adata_itn.obs['patient_ID'].nunique())

In [ ]:
def evaluate_models_on_test_with_roc(train_results_dir, adata_test, verbose=True):
    models_path = os.path.join(train_results_dir, 'models.joblib')
    feature_names_path = os.path.join(train_results_dir, 'feature_names.txt')

    if verbose:
        print(f"Loading models from: {models_path}")
        print(f"Loading features from: {feature_names_path}")

    models = load(models_path)
    with open(feature_names_path, 'r') as f:
        feature_names = f.read().splitlines()

    train_results_df = pd.read_csv(os.path.join(train_results_dir, 'results.csv'))
    if 'optimal_threshold' not in train_results_df.columns:
        raise ValueError(f"'optimal_threshold' not found in {train_results_dir}/results.csv")

    if verbose:
        print(f"\nPreparing test data...")
        print(f"Using layer: log_fold_change_over_AG")

    X_test = adata_test.to_df(layer="log_fold_change_over_AG")
    X_test = X_test[feature_names]
    X_test[np.isnan(X_test) | np.isinf(X_test)] = 0

    if verbose:
        print("\nLabel distribution:")
        print(adata_test.obs['group'].value_counts(dropna=False))

    y_test = (adata_test.obs['group'] == 'lupus').astype(int)

    results, all_predictions, roc_data = [], [], []
    all_thresholds = []
    predictions_data = {"y_true": y_test.values}

    if verbose:
        print(f"\nEvaluating {len(models)} models on test set...")

    for model_name, model in models.items():
        pred_proba = model.predict_proba(X_test)[:, 1]
        all_predictions.append(pred_proba)

        fpr, tpr, thresholds = roc_curve(y_test, pred_proba)
        roc_data.append({
            'model': model_name, 'fpr': fpr, 'tpr': tpr,
            'thresholds': thresholds, 'y_true': y_test.values,
            'y_pred_proba': pred_proba})

        seed = int(model_name.split('_')[1])
        fold = int(model_name.split('_')[3])

        matching_rows = train_results_df[
            (train_results_df['seed'] == seed) &
            (train_results_df['fold'] == fold)]
        optimal_threshold = matching_rows['optimal_threshold'].values[0]
        all_thresholds.append(optimal_threshold)

        pred_binary = (pred_proba >= optimal_threshold).astype(int)

        predictions_data[f"{model_name}_proba"] = pred_proba
        predictions_data[f"{model_name}_label"] = pred_binary

        tn, fp, fn, tp = confusion_matrix(y_test, pred_binary).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0
        f1 = f1_score(y_test, pred_binary)
        balanced_acc = balanced_accuracy_score(y_test, pred_binary)

        auroc = roc_auc_score(y_test, pred_proba)
        auprc = average_precision_score(y_test, pred_proba)

        results.append({
            'model': model_name, 'seed': seed, 'fold': fold,
            'auroc': auroc, 'auprc': auprc,
            'optimal_threshold': optimal_threshold,
            'sensitivity': sensitivity, 'specificity': specificity,
            'ppv': ppv, 'npv': npv, 'f1_score': f1,
            'balanced_accuracy': balanced_acc,
            'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp})

    results_df = pd.DataFrame(results)

    ensemble_pred_proba = np.mean(all_predictions, axis=0)
    ensemble_threshold = np.mean(all_thresholds)
    ensemble_threshold_std = np.std(all_thresholds)

    if verbose:
        print(f"\nEnsemble threshold (from training): {ensemble_threshold:.3f} ± {ensemble_threshold_std:.3f}")

    ensemble_pred_binary = (ensemble_pred_proba >= ensemble_threshold).astype(int)
    predictions_data["ensemble_proba"] = ensemble_pred_proba
    predictions_data["ensemble_label"] = ensemble_pred_binary
    predictions_df = pd.DataFrame(predictions_data, index=X_test.index)

    ensemble_cm = confusion_matrix(y_test, ensemble_pred_binary)
    tn, fp, fn, tp = ensemble_cm.ravel()
    ensemble_fpr, ensemble_tpr, ensemble_thresholds = roc_curve(y_test, ensemble_pred_proba)

    ensemble_metrics = {
        'auroc': roc_auc_score(y_test, ensemble_pred_proba),
        'auprc': average_precision_score(y_test, ensemble_pred_proba),
        'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'ppv': tp / (tp + fp) if (tp + fp) > 0 else 0,
        'npv': tn / (tn + fn) if (tn + fn) > 0 else 0,
        'f1_score': f1_score(y_test, ensemble_pred_binary),
        'balanced_accuracy': balanced_accuracy_score(y_test, ensemble_pred_binary),
        'confusion_matrix': ensemble_cm,
        'optimal_threshold': ensemble_threshold,
        'optimal_threshold_std': ensemble_threshold_std,
        'fpr': ensemble_fpr, 'tpr': ensemble_tpr, 'thresholds': ensemble_thresholds}

    if verbose:
        print("\n" + "="*60)
        print("ENSEMBLE MODEL PERFORMANCE:")
        print("="*60)
        print(f"Ensemble Threshold: {ensemble_threshold:.3f} ± {ensemble_threshold_std:.3f}")
        print(f"AUROC: {ensemble_metrics['auroc']:.3f}")
        print(f"AUPRC: {ensemble_metrics['auprc']:.3f}")
        for metric in ['sensitivity', 'specificity', 'ppv', 'npv', 'f1_score', 'balanced_accuracy']:
            print(f"{metric.upper()}: {ensemble_metrics[metric]:.3f}")
        print("\nEnsemble Confusion Matrix:")
        print(ensemble_metrics['confusion_matrix'])
        print(f"\nTotal test samples: {len(X_test)}")
        print(f"Total test donors: {adata_test.obs['patient_ID'].nunique()}")

    return {
        'results_df': results_df, 'ensemble_metrics': ensemble_metrics,
        'predictions_df': predictions_df, 'roc_data': roc_data}

In [ ]:
def plot_test_roc_curves(test_results, train_results_dir, save_plots=True, show_individual=False):
    results_df = test_results['results_df']
    roc_data = test_results['roc_data']

    fig, ax = plt.subplots(figsize=(9, 4.5))

    if show_individual:
        for data in roc_data:
            ax.plot(data['fpr'], data['tpr'], alpha=0.1, color='blue', linewidth=0.8)

    mean_fpr = np.linspace(0, 1, 1000)
    tprs = []
    for data in roc_data:
        interp_tpr = np.interp(mean_fpr, data['fpr'], data['tpr'])
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)

    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    std_tpr = np.std(tprs, axis=0)

    mean_auc = results_df['auroc'].mean()
    std_auc = results_df['auroc'].std()

    ax.plot(mean_fpr, mean_tpr, color='darkblue', linewidth=3,
            label=f'Mean ROC (AUC = {mean_auc:.2f} ± {std_auc:.2f})')

    tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
    tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
    ax.fill_between(mean_fpr, tprs_lower, tprs_upper,
                    color='blue', alpha=0.2, label='± 1 std. dev.')

    ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random classifier')
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    ax.set_xlabel('False Positive Rate', fontsize=14)
    ax.set_ylabel('True Positive Rate', fontsize=14)
    ax.legend(loc='lower right', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal', adjustable='box')
    plt.tight_layout()

    if save_plots:
        output_path = os.path.join(train_results_dir, 'itn_test_roc_curves.png')
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"\nROC curves saved to: {output_path}")

    plt.show()
    return fig, ax

def run_test_evaluation(train_results_dir, adata_test, save_results=True):
    print(f"\n{'='*60}")
    print(f"EVALUATING ITN TEST PERFORMANCE")
    print(f"Training results: {train_results_dir}")
    print(f"{'='*60}\n")

    test_results = evaluate_models_on_test_with_roc(train_results_dir, adata_test)
    plot_test_roc_curves(test_results, train_results_dir, save_plots=save_results, show_individual=False)

    if save_results:
        results_path = os.path.join(train_results_dir, 'itn_test_results.csv')
        test_results['results_df'].to_csv(results_path, index=False)
        print(f"\nTest results saved to: {results_path}")

        predictions_path = os.path.join(train_results_dir, 'itn_test_predictions.csv')
        test_results['predictions_df'].to_csv(predictions_path)
        print(f"Predictions saved to: {predictions_path}")

        roc_path = os.path.join(train_results_dir, 'itn_test_roc_data.joblib')
        dump(test_results['roc_data'], roc_path)
        print(f"ROC data saved to: {roc_path}")

        ensemble_path = os.path.join(train_results_dir, 'itn_test_ensemble_metrics.joblib')
        dump(test_results['ensemble_metrics'], ensemble_path)
        print(f"Ensemble metrics saved to: {ensemble_path}")

    return test_results

In [ ]:
test_results = run_test_evaluation(MODEL_DIR, adata_itn, save_results=True)